# Script 11 - Age moderation, WLS robustness at finer age resolution

Re-estimates the age moderation of the rhythm-outcome paths with
survey-weighted WLS and HC3 errors. The inferential age-moderation result is
in the Mplus interaction models (`x_age_*`, design-corrected; reported in
`SuppFigS5_Age_Moderation_SimpleSlopes` and the moderation sheet of
`Supp_Tables.xlsx`); this notebook adds four age strata
(20-39 / 40-59 / 60-69 / 70+) and a second estimator.

All three rhythm metrics (intradaily variability, interdaily stability,
relative amplitude) are tested on the two mediator paths (rhythm to KDM-BA
advancement, rhythm to log SII) and on the two outcomes (PHQ-9 in adults 20+,
cognition composite in 60+).

Estimates carry no complex-survey design correction, so intervals are
narrower than the design-corrected ones. Inference follows the Mplus models;
no number from this notebook goes into the main text.

Reads `data/processed/8_nhanes_analytic.csv`.
Writes `result/supplementary/SuppTableS8_Age_Moderation_WLS.xlsx`,
`result/raw/script11_age_moderation_full.csv`,
`figure/supplementary/SuppFigS1_Age_Moderation_WLS_Forest.{pdf,png}` and
`figure/source_data/Source_Data_SuppFigS1_Age_Moderation_WLS_Forest.xlsx`.


## Step 0 - Locate project, imports

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE   = find_root()
PROC   = BASE / 'data' / 'processed'
R_SUPP = BASE / 'result' / 'supplementary'
R_RAW  = BASE / 'result' / 'raw'
F_SUPP = BASE / 'figure' / 'supplementary'
F_SRC  = BASE / 'figure' / 'source_data'
for q in (R_SUPP, R_RAW, F_SUPP, F_SRC):
    q.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import nature_palette
importlib.reload(nature_palette)
import wls_utils
import population; importlib.reload(population)
for _m in (wls_utils, nature_palette,):
    importlib.reload(_m)

from wls_utils import wls_fit
from population import ADULT_MIN_AGE, OLD_MIN_AGE, YOUNG, OLD, POOLED
from nature_palette import (PHQ9_COLOR, COG_COLOR, KDM_BA_COLOR, SII_COLOR,
                            GREY, BLACK)
from nature_palette import fmt_ci as _fmt_ci, age_label

# One colour per rhythm metric, shared with the main figures.
from nature_palette import EXP_COL, EXP_ORDER  # one source of truth
EXP_NAME = {'IV': 'Intradaily variability', 'IS': 'Interdaily stability',
            'RA': 'Relative amplitude'}
EXPOSURES = list(EXP_ORDER)


## Step 1 - Load; stored weight; PHQ-9 / cognition composite; strata

In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv', low_memory=False)
rename = {
    'RIDAGEYR':       'AGE',
    'RIAGENDR':       'SEX',
    'BMXBMI':         'BMI',
    'DMDEDUC2':       'EDU',
    'Wear_Time':      'WEAR_T',
    'SLD010H':        'SLEEP_H',
    'KDM_BA_advance': 'KDM_BA_ADV',
    'SII_log':        'SII_LOG',
}
df = df.rename(columns=rename)

# The four-year MEC weight WTMEC4YR = WTMEC2YR / 2 is computed once in
# script 8 and stored; assert rather than recompute.
assert 'WTMEC4YR' in df.columns, 'run script 8: WTMEC4YR missing from the analytic file'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'

df = df[df['AGE'] >= ADULT_MIN_AGE].copy()   # the analysis population (population.py)

# PHQ-9 total (canonical: all nine items answered)
dpq_cols = [f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]
df['PHQ9'] = df[dpq_cols].replace({7: np.nan, 9: np.nan}).sum(axis=1, min_count=9)

# Cognition composite z (within-60+ z-mean; a score of 0 is a valid result and is kept)
elig_60 = df['AGE'] >= OLD_MIN_AGE
for src, dst in [('memory_score', 'MEM_Z'),
                 ('fluency_score', 'FLUEN_Z'),
                 ('speed_score',  'SPEED_Z')]:
    raw = df[src]
    s = raw.loc[elig_60].dropna()
    df[dst] = np.where(elig_60, (raw - s.mean()) / s.std(), np.nan)
df['COG_Z'] = df[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

def age_stratum(a):
    if pd.isna(a) or a < ADULT_MIN_AGE: return None
    if a < 40: return f'{ADULT_MIN_AGE}-39'
    if a < 60: return '40-59'
    if a < 70: return '60-69'
    return '70+'
df['Age_stratum'] = df['AGE'].apply(age_stratum)

print(f'Adults {ADULT_MIN_AGE}+: n = {len(df):,}')
for c in ['IV', 'IS', 'RA', 'KDM_BA_ADV', 'SII_LOG', 'PHQ9', 'COG_Z']:
    print(f'  {c:11s} non-null: {df[c].notna().sum():,}')


## Step 2 - Helpers, parameterised by rhythm metric

In [ ]:
M1_COVS = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
INT_COVS = [c for c in M1_COVS if c != 'AGE']   # AGE enters the interaction model as a z term

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def fmt_ci(est, lo, hi, dp=3):
    return _fmt_ci(est, lo, hi, dp)   # minus kept, plus omitted

def weighted_mean_sd(x, w):
    w_tot = np.sum(w)
    mu = np.sum(x * w) / w_tot
    var = np.sum(w * (x - mu) ** 2) / w_tot
    return mu, float(np.sqrt(var))

def per_outcome_sample(exposure, outcome_col, age_min=ADULT_MIN_AGE, age_max=999):
    needed = [exposure, outcome_col, 'WTMEC4YR'] + M1_COVS
    mask = (df['AGE'] >= age_min) & (df['AGE'] < age_max) \
        & df[needed].notna().all(axis=1)
    return df.loc[mask].copy()

def run_interaction(exposure, outcome_col, age_min=ADULT_MIN_AGE, age_max=999):
    """outcome_z ~ exp_z + AGE_z + exp_z x AGE_z + covariates, WLS-HC3; the
    interaction coefficient is the shift in the STDYX exposure->outcome
    coefficient per 1 s.d. of age."""
    ana = per_outcome_sample(exposure, outcome_col, age_min, age_max)
    w = ana['WTMEC4YR'].values
    mu_e, sd_e = weighted_mean_sd(ana[exposure].values,    w)
    mu_a, sd_a = weighted_mean_sd(ana['AGE'].values,       w)
    mu_y, sd_y = weighted_mean_sd(ana[outcome_col].values, w)
    e_z = (ana[exposure].values    - mu_e) / sd_e
    a_z = (ana['AGE'].values       - mu_a) / sd_a
    y_z = (ana[outcome_col].values - mu_y) / sd_y
    X = np.column_stack([e_z, a_z, e_z * a_z]
                        + [ana[c].values for c in INT_COVS])
    fit = wls_fit(X, y_z, w=w, se='hc3', add_const=True)
    out = {'n': int(fit['n']), 'r2': float(fit['r2'])}
    for i, name in enumerate(['EXP', 'AGE', 'EXP_x_AGE'], start=1):
        out[name] = {'beta': float(fit['params'][i]),
                     'ci_lo': float(fit['ci_lo'][i]),
                     'ci_hi': float(fit['ci_hi'][i]),
                     'p': float(fit['p'][i])}
    return out

def run_stratum_M1(exposure, outcome_col, age_min, age_max, stratum_label):
    """outcome_z ~ exp_z + covariates within one age stratum, WLS-HC3."""
    ana = per_outcome_sample(exposure, outcome_col, age_min, age_max)
    if len(ana) < 50:
        return {'stratum': stratum_label, 'n': len(ana), 'beta': np.nan,
                'ci_lo': np.nan, 'ci_hi': np.nan, 'p': np.nan, 'r2': np.nan}
    w = ana['WTMEC4YR'].values
    mu_e, sd_e = weighted_mean_sd(ana[exposure].values,    w)
    mu_y, sd_y = weighted_mean_sd(ana[outcome_col].values, w)
    e_z = (ana[exposure].values    - mu_e) / sd_e
    y_z = (ana[outcome_col].values - mu_y) / sd_y
    X = np.column_stack([e_z] + [ana[c].values for c in M1_COVS])
    fit = wls_fit(X, y_z, w=w, se='hc3', add_const=True)
    return {'stratum': stratum_label, 'n': int(fit['n']),
            'beta':  float(fit['params'][1]),
            'ci_lo': float(fit['ci_lo'][1]),
            'ci_hi': float(fit['ci_hi'][1]),
            'p':     float(fit['p'][1]),
            'r2':    float(fit['r2'])}


## Step 3 - Continuous age x metric interactions (3 metrics x 4 outcomes)

In [ ]:
# Continuous age x metric interaction, every rhythm metric, four outcomes.
# The two mediator paths are included because the age moderation acts on the
# rhythm-to-mediator leg.
OUTCOMES = [
    ('KDM_BA_ADV', f'KDM-BA advancement (a1 path, {POOLED})', ADULT_MIN_AGE, 999),
    ('SII_LOG',    f'log SII (a2 path, {POOLED})',            ADULT_MIN_AGE, 999),
    ('PHQ9',       f'Depression (PHQ-9, {POOLED})',           ADULT_MIN_AGE, 999),
    ('COG_Z',      f'Cognition (composite z, {OLD})',       OLD_MIN_AGE, 999),
]

interaction_results = {}
for exp in EXPOSURES:
    for col, label, lo, hi in OUTCOMES:
        interaction_results[(exp, col)] = run_interaction(exp, col, lo, hi)

print(f'{"":24s}' + ''.join(f'{e:>22s}' for e in EXPOSURES))
for col, label, lo, hi in OUTCOMES:
    cells = []
    for exp in EXPOSURES:
        r = interaction_results[(exp, col)]['EXP_x_AGE']
        star = '*' if r['p'] < 0.05 else ' '
        cells.append(f"{r['beta']:+.3f} (p={fmt_p(r['p'])}){star}")
    print(f'{label[:24]:24s}' + ''.join(f'{c:>22s}' for c in cells))
print('\n(interaction coefficients; * p < 0.05. Estimates are WLS-HC3 '
      'without the design correction -- inference follows Mplus.)')


## Step 4 - Age-stratified estimates

In [ ]:
PHQ_STRATA = [(f'{ADULT_MIN_AGE}-39', ADULT_MIN_AGE, 40), ('40-59', 40, 60),
              ('60-69', 60, 70), ('70+', 70, 999)]
COG_STRATA = [('60-69', 60, 70), ('70+', 70, 999)]
# The 20-59 / 60+ pair mirrors the Mplus age cells; the four narrower bands
# locate where a slope changes sign.
MED_STRATA = PHQ_STRATA + [(YOUNG, ADULT_MIN_AGE, OLD_MIN_AGE), (OLD, OLD_MIN_AGE, 999)]

stratum_results = {}
for exp in EXPOSURES:
    stratum_results[(exp, 'KDM_BA_ADV')] = [
        run_stratum_M1(exp, 'KDM_BA_ADV', lo, hi, lab) for lab, lo, hi in MED_STRATA]
    stratum_results[(exp, 'SII_LOG')] = [
        run_stratum_M1(exp, 'SII_LOG', lo, hi, lab) for lab, lo, hi in MED_STRATA]
    stratum_results[(exp, 'PHQ9')] = [
        run_stratum_M1(exp, 'PHQ9', lo, hi, lab) for lab, lo, hi in PHQ_STRATA]
    stratum_results[(exp, 'COG_Z')] = [
        run_stratum_M1(exp, 'COG_Z', lo, hi, lab) for lab, lo, hi in COG_STRATA]

for exp in EXPOSURES:
    print(f'\n{EXP_NAME[exp]} -> KDM-BA advancement (a1) by age stratum:')
    for r in stratum_results[(exp, 'KDM_BA_ADV')]:
        print(f'  {r["stratum"]:6s}  n={r["n"]:>5,}  beta={r["beta"]:+.3f}  '
              f'CI=({r["ci_lo"]:+.3f}, {r["ci_hi"]:+.3f})  p={fmt_p(r["p"])}')


## Step 5 - Supp Table S8

In [ ]:
rows = []
for exp in EXPOSURES:
    rows.append({'Block': f'{EXP_NAME[exp]} ({exp})', 'Analysis': '', 'n': '',
                 'beta_STDYX (95% CI)': '', 'p': ''})
    for col, label, lo, hi in OUTCOMES:
        R = interaction_results[(exp, col)]
        rows.append({'Block': '', 'Analysis': label, 'n': f"{R['n']:,}",
                     'beta_STDYX (95% CI)': '', 'p': ''})
        for name, pretty in [('EXP', f'  {exp} at mean age'),
                             ('AGE', '  age main effect'),
                             ('EXP_x_AGE', f'  {exp} x age interaction')]:
            r = R[name]
            rows.append({'Block': '', 'Analysis': pretty, 'n': '',
                         'beta_STDYX (95% CI)': fmt_ci(r['beta'], r['ci_lo'], r['ci_hi']),
                         'p': fmt_p(r['p'])})
        skey = (exp, col)
        if skey in stratum_results:
            rows.append({'Block': '', 'Analysis': '  age-stratified estimates',
                         'n': '', 'beta_STDYX (95% CI)': '', 'p': ''})
            for s in stratum_results[skey]:
                if pd.isna(s['beta']):
                    continue
                _band = age_label(s['stratum']).replace('70+', '\u226570')
                rows.append({'Block': '', 'Analysis': f"    {_band}",
                             'n': f"{s['n']:,}",
                             'beta_STDYX (95% CI)': fmt_ci(s['beta'], s['ci_lo'], s['ci_hi']),
                             'p': fmt_p(s['p'])})

t = pd.DataFrame(rows)
print(f'{len(t)} rows')

NOTES = [
    'Survey-weighted WLS with HC3 errors; NO complex-survey design correction. '
    'Inference follows the Mplus models (Supp Tables S3-S7); this table is a '
    'robustness check at finer age resolution, and no number from it appears '
    'in the main text.',
    'Exposure and outcome z-standardised with survey-weighted mean and s.d. on '
    'each analytic sample; covariates on raw scales.',
    'Interaction model: outcome_z ~ metric_z + age_z + metric_z x age_z + sex, '
    'BMI, education, wear time, sleep duration. The interaction coefficient is '
    'the shift in the STDYX metric->outcome coefficient per 1 s.d. of age.',
    'Stratified model: outcome_z ~ metric_z + core covariates (including '
    'continuous age) within each stratum.',
    'The mediator paths (KDM-BA advancement, log SII) are tested as outcomes '
    'because the age moderation lives on the rhythm->mediator leg, not on the '
    'outcome leg.',
    'Cognition is defined only at 60+, where NHANES administers the battery; '
    'its strata are 60-69 and 70+.',
]
t.to_csv(R_RAW / 'script11_age_moderation_full.csv', index=False)
with pd.ExcelWriter(R_SUPP / 'SuppTableS8_Age_Moderation_WLS.xlsx',
                    engine='openpyxl') as xw:
    t.to_excel(xw, sheet_name='Supp Table S8', index=False)
    pd.DataFrame({'Notes': NOTES}).to_excel(xw, sheet_name='Notes', index=False)
print(f"wrote {R_SUPP / 'SuppTableS8_Age_Moderation_WLS.xlsx'}")
print(f"wrote {R_RAW / 'script11_age_moderation_full.csv'}")


## Step 6 - Supp Figure S1: the a1 path by age stratum, all three metrics

One panel per rhythm metric; the four age bands on the y axis, the stratified
rhythm-to-KDM-BA-advancement coefficient on x.


In [ ]:
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.linewidth': 0.5, 'lines.linewidth': 0.9,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600,
    'axes.spines.top': False, 'axes.spines.right': False,
})
MM = 1 / 25.4
BANDS = [f'{ADULT_MIN_AGE}-39', '40-59', '60-69', '70+']   # keys of stratum_results
BAND_LABEL = {b: ('\u226570' if b == '70+' else b.replace('-', '\u2013')) for b in BANDS}

fig, axes = plt.subplots(1, 3, figsize=(183 * MM, 60 * MM), sharey=True)
src = []
for ax, exp in zip(axes, EXPOSURES):
    res = {r['stratum']: r for r in stratum_results[(exp, 'KDM_BA_ADV')]}
    for y, band in enumerate(reversed(BANDS)):
        r = res[band]
        sig = r['p'] < 0.05
        col = EXP_COL[exp]      # hue = metric, always; fill = significance
        ax.errorbar(r['beta'], y,
                    xerr=[[r['beta'] - r['ci_lo']], [r['ci_hi'] - r['beta']]],
                    fmt='o', ms=3.6, color=col, mfc=col if sig else 'white',
                    ecolor=col, elinewidth=0.9, capsize=1.8, mew=0.9)
        src.append(dict(metric=exp, stratum=band, n=r['n'], beta=r['beta'],
                        ci_lo=r['ci_lo'], ci_hi=r['ci_hi'], p=r['p']))
    ax.axvline(0, color='0.4', lw=0.5, ls=(0, (3, 2)))
    ax.set_yticks(range(len(BANDS)))
    ax.set_yticklabels([BAND_LABEL[b] for b in reversed(BANDS)])
    ax.set_title(f'{EXP_NAME[exp]} ({exp})', color=EXP_COL[exp], pad=3)
    ax.set_xlabel('Standardized coefficient \u03b2 (95% CI)')
axes[0].set_ylabel('Age group')
fig.tight_layout()
for ax, letter in zip(axes, 'abc'):
    fig.text(ax.get_position().x0 - 0.02, 0.98, letter, ha='right', va='top',
             fontsize=10, fontweight='bold')

stem = 'SuppFigS1_Age_Moderation_WLS_Forest'
fig.savefig(F_SUPP / f'{stem}.pdf', bbox_inches='tight')
fig.savefig(F_SUPP / f'{stem}.png', bbox_inches='tight', dpi=600)
plt.close(fig)

cap = ['Supplementary Fig. S1 | Rest-activity rhythm associations with KDM '
       'biological-age advancement within four age groups.',
       f'a, relative amplitude (RA); b, interdaily stability (IS); c, intradaily '
       f'variability (IV); age groups {BAND_LABEL[BANDS[0]]}, 40\u201359, 60\u201369 and \u226570 years.',
       'Points are standardized coefficients from survey-weighted least-squares '
       'models with HC3 standard errors fitted within each age group, with the '
       'metric and biological-age advancement standardized by their weighted '
       'mean and standard deviation in that group; they are not Mplus STDYX '
       'coefficients. Bars are 95% confidence intervals.',
       'Open markers indicate P \u2265 0.05; P values and intervals are not '
       'adjusted for multiple comparisons. Differences between age groups are '
       'assessed by the interaction models of the main text (Table 3), not by '
       'comparing these estimates.',
       'Covariates: age (continuous, within group), sex, body mass index, '
       'education, accelerometer wear time and sleep duration. No complex-survey '
       'design correction; supportive analysis, estimates in Supplementary Table 8.']
with pd.ExcelWriter(F_SRC / f'Source_Data_{stem}.xlsx', engine='openpyxl') as xw:
    pd.DataFrame(src).to_excel(xw, sheet_name='source data', index=False)
    pd.DataFrame({'Notes': cap}).to_excel(xw, sheet_name='legend', index=False)
print(f'wrote {F_SUPP / stem}.pdf + .png')
print(f'wrote {F_SRC / ("Source_Data_" + stem + ".xlsx")}')

# Move outputs under superseded file names to _superseded/.
for stale in [R_SUPP / 'SuppTableS3_Age_Moderation.xlsx',
              F_SUPP / 'SuppFigS1_Age_Moderation_Forest.png',
              F_SUPP / 'SuppFigS1_Age_Moderation_Forest.tiff',
              F_SRC / 'Source_Data_SuppFigS1.xlsx']:
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')
